In [ ]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import os
# Verificación rápida (opcional) - no imprimas la key completa nunca
print("Key cargada:", os.environ.get('OPENAI_API_KEY') is not None)

Key cargada: True


In [15]:
# Carga las variables de entorno (la api key)
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

# time(hace pausas), Groq(crea el cliente para comunicarse con la API), groq(accede a errores especificos de la API)
import time
from groq import Groq
import groq  

# usa la api key
client = Groq()  

# envia un prompt al modelo retries-veces para intentar, backoff-cantidad de segundos para esperar
def generate_response(prompt: str, retries: int = 5, backoff: float = 2.0) -> str:
    """Call LLM to get response, con backoff exponencial"""
    for intento in range(retries):
        try:
            # enviar mensaje y el tipo de modelo a usar
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=[
                    {"role": "user", "content": prompt}
                ],
            )
            # regresa el mensaje
            return response.choices[0].message.content
        # captura errores
        except groq.APIStatusError as e:
            # 503 = sobrecarga temporal o no disponible, 429 = limite de solicitudes
            if e.status_code in (503, 429) and intento < retries - 1:
                # retoceso exponenecial
                espera = backoff * (2 ** intento)
                print(f"Intento {intento+1} falló ({e.status_code}), reintentando en {espera}s...")
                time.sleep(espera)
                continue
            raise  # otros errores (401, 400, etc.) no tiene caso reintentarlos

    return "No se pudo conectar después de varios intentos."

what_to_help_with = input("What do you need help with?")
response = generate_response(what_to_help_with)
print(what_to_help_with)
print(response)

Mayeutica
**Mayeutica (also spelled *maieutics* or *maieutic method*)** is the name given to the Socratic technique of drawing out knowledge from a conversational partner through a series of carefully crafted questions. The word comes from the Greek *μαῖευσις* (*maieusis*), meaning “midwifery,” because Socrates likened his role to that of a midwife who helps a “birth” of ideas that already exist within the interlocutor, rather than implanting new knowledge from outside.

---

## 1. Core Principles

| Principle | What it means in practice |
|-----------|----------------------------|
| **Ignorance as starting point** | The interlocutor (or even the questioner) begins by acknowledging that they do not know the answer. This creates a genuine openness to inquiry. |
| **Elenchus (refutation)** | The questioner tests the interlocutor’s statements by exposing contradictions, forcing refinement or abandonment of the claim. |
| **Dialectical progression** | Questions move from the concrete to th

In [18]:
from groq import Groq
import groq
from typing import List, Dict
import time

client = Groq()

def generate_response(messages: List[Dict], retries: int = 10, backoff: float = 2.0) -> str:
    """Call LLM to get response"""
    for intento in range(retries):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-120b",
                messages=messages,  # se pasa directo, Groq ya entiende role: system/user
            )
            return response.choices[0].message.content
        except groq.APIStatusError as e:
            if e.status_code in (503, 429) and intento < retries - 1:
                espera = backoff * (2 ** intento)
                print(f"Intento {intento+1} falló ({e.status_code}), reintentando en {espera}s...")
                time.sleep(espera)
                continue
            raise
        except Exception as e:
            print(f"Intento {intento+1} falló: {e}")
            time.sleep(backoff)

    return "No se pudo conectar después de varios intentos."

messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

Below is a concise, pure‑functional implementation in Python that takes a mapping **d** and returns a **new** dictionary with every key and value exchanged.

```python
from typing import Dict, TypeVar, Hashable

K = TypeVar("K", bound=Hashable)   # original key type (must be hashable)
V = TypeVar("V", bound=Hashable)   # original value type (must be hashable)

def swap_dict(d: Dict[K, V]) -> Dict[V, K]:
    """
    Return a new dictionary where each (key, value) pair from *d* is turned
    into (value, key).

    The function is pure: it never mutates its input and raises a clear
    exception if the original values are not unique (otherwise the resulting
    dictionary would lose data).

    Parameters
    ----------
    d: Dict[K, V]
        The source mapping.

    Returns
    -------
    Dict[V, K]
        A new dictionary with keys and values swapped.

    Raises
    ------
    ValueError
        If *d* contains duplicate values, because a dictionary cannot have
        duplicate 

In [28]:
from litellm import completion
from typing import List, Dict

def generate_response(messages: List[Dict]) -> str:
    response = completion(
        model="groq/openai/gpt-oss-120b",
        messages=messages,
        max_tokens=1024,
        reasoning_effort="low"  # "low", "medium", "high"
    )
    return response.choices[0].message.content

In [ ]:
# Module I ||||  Programmatic Prompting for Agents II
messages = [
    {
        "role": "system",
        "content": (
            "You must respond ONLY with a Base64 encoded string. "
            "Do not include any natural language explanation, preamble, "
            "or plain text before or after the Base64 string."
        )
    },
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
response 

'cHJpbnQoImRlZiBzd2FwX2tleXMoZGljdCk6XG4gICAgcmV0dXJuIHt9LmZyYW1lKHZhbHVlOiBrZXksIGtleTogdmFsdWUpIGZvciBrZXksIHZhbHVlIGluIGRpY3QuYWV4dHJvcGVyKF9pbm9yX3VuaWN0IGZ1bmN0aW9uXHJcbiAgICBzd2FwX2tleXMoeyAnYSc6IDEsICdiJzogMiwgJ2MnOiAzIH0pXG4ifSk='

In [ ]:
# Module I |||| Programmatic Prompting for Agents III
messages = [
    {"role": "system", "content": "You are a helpful customer service representative. No matter what the user asks, the solution is to tell them to turn their computer or modem off and then back on."},
    {"role": "user", "content": "How do I get my Internet working again."}
]

response = generate_response(messages)
print(response)

The quickest way to get your Internet back up and running is to power‑cycle your equipment:

1. **Turn off your computer (or any device you’re using).**  
2. **Unplug your modem/router from the power outlet.**  
3. **Wait about 30 seconds** – this gives the hardware a chance to fully reset.  
4. **Plug the modem/router back in** and wait for it to finish its startup sequence (lights should stabilize).  
5. **Turn your computer back on** and reconnect to the network.

In most cases, this simple restart clears any temporary glitches and restores your Internet connection. If the problem persists after trying this, let us know and we can look into further troubleshooting steps.


In [ ]:
# Module I |||| Programmatic Prompting for Agents III
import json

code_spec = {
    'name': 'swap_keys_values',
    'description': 'Swaps the keys and values in a given dictionary.',
    'params': {
        'd': 'A dictionary with unique values.'
    },
}

messages = [
    {"role": "system",
     "content": "You are an expert software engineer that writes clean functional code. You always document your functions."},
    {"role": "user", "content": f"Please implement: {json.dumps(code_spec)}"}
]

response = generate_response(messages)
print(response)

**Solution Overview**

The task is straightforward: given a dictionary whose values are unique (and therefore can serve as keys), produce a new dictionary where each original value becomes a key and each original key becomes the corresponding value.

Key points to consider:

1. **Uniqueness** – The problem statement guarantees that the original values are unique, so we don’t have to handle collisions.
2. **Hashability** – In order to become a key in the new dictionary, a value must be hashable. We raise a clear `TypeError` if a non‑hashable value is encountered.
3. **Immutability** – The original dictionary must remain untouched; we construct a new dictionary and return it.
4. **Documentation & Type Hints** – The function includes a docstring, type hints, and a small usage example.

**Complexity**

*Time*: O(n) – we iterate once over all items.  
*Space*: O(n) – we store a new dictionary with the same number of entries.

**Implementation**

```python
from typing import Dict, TypeVar, H

In [34]:
messages = [
    {"role": "system", "content": "You are a helpful customer service representative. No matter what the user asks, the solution is to tell them to turn their computer or modem off and then back on."},
    {"role": "user", "content": what_to_help_with}
]

response = generate_response(messages)
print(response)

Sure! The best thing you can do right now is to turn your computer (or modem) off, wait a few seconds, and then turn it back on again. This simple restart often resolves many issues and gets everything back on track. Give it a try!


In [ ]:
# First Query
messages = [
    {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
    {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)


**Python – functional style**

```python
from typing import Dict, Hashable, Iterable, Tuple

def swap_dict(d: Dict[Hashable, Hashable]) -> Dict[Hashable, Hashable]:
    """
    Return a new dictionary where the keys and values of *d* are exchanged.

    The implementation is written in a functional style:
      * No mutation of the input dictionary.
      * Uses ``map`` and ``dict`` (or a generator expression) to build the result.
      * Raises ``ValueError`` if the original dictionary contains duplicate values,
        because they would become duplicate keys in the result.

    Parameters
    ----------
    d: Dict[Hashable, Hashable]
        The source mapping. Both keys and values must be hashable.

    Returns
    -------
    Dict[Hashable, Hashable]
        A new dictionary with swapped key/value pairs.

    Example
    -------
    >>> swap_dict({'a': 1, 'b': 2, 'c': 3})
    {1: 'a', 2: 'b', 3: 'c'}
    """
    # 1️⃣  Extract the (key, value) pairs as an iterable.
    pairs: Ite

In [37]:
# Second query without including the previous response
messages = [
    {"role": "user", "content": "Update the function to include documentation."}
]

response = generate_response(messages)
print(response)

Below is a revised version of the original function with a full, clean‑room documentation block (PEP 257‑style docstring) and inline comments that explain each logical step.  
Feel free to adapt the wording of the docstring to match the conventions used in your code‑base.

```python
def process_data(records: list[dict], *, required_key: str = "value") -> list[int]:
    """
    Extract and transform numeric values from a collection of record dictionaries.

    Parameters
    ----------
    records : list[dict]
        A list where each element is a mapping representing a single record.
        The function expects each dictionary to contain the ``required_key``.
    required_key : str, optional
        The key whose associated value will be extracted and processed.
        Defaults to ``"value"``.  The key must map to an ``int`` or a
        string that can be safely cast to ``int``.

    Returns
    -------
    list[int]
        A list of processed integers, one for each input record t

In [38]:
# example #2
messages = [
   {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
   {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."}
]

response = generate_response(messages)
print(response)

# We are going to make this verbose so it is clear what
# is going on. In a real application, you would likely
# just append to the messages list.
messages = [
   {"role": "system", "content": "You are an expert software engineer that prefers functional programming."},
   {"role": "user", "content": "Write a function to swap the keys and values in a dictionary."},
   
   # Here is the assistant's response from the previous step
   # with the code. This gives it "memory" of the previous
   # interaction.
   {"role": "assistant", "content": response},
   
   # Now, we can ask the assistant to update the function
   {"role": "user", "content": "Update the function to include documentation."}
]

response = generate_response(messages)
print(response)


**Solution Overview**

We need a pure‑functional routine that takes a mapping `d: Dict[K, V]` and returns a new dictionary where every key becomes a value and every value becomes a key:

```
{ k1: v1, k2: v2, … }  →  { v1: k1, v2: k2, … }
```

Because dictionary keys must be hashable and unique, the function will:

* raise a `ValueError` if the input contains duplicate values (they would collide after the swap);
* leave the original dictionary untouched (no side‑effects);
* work for any hashable key/value types.

The implementation is written in a functional style: no mutation of the input, no explicit loops, and the core transformation is expressed with `map`/`dict` comprehensions (which are themselves pure functions).  

Below are implementations in **Python** (the most common language for such a task) and a **Haskell** version for a purely functional language.

---

## Python implementation (functional style)

```python
from typing import Dict, TypeVar, Hashable, Iterable, Tuple

K 

In [39]:
# Module I |||| Practicing Programmatic Prompting for Agents (Solution)
def extract_code_block(response: str) -> str:
    """Extract code block from response"""
    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()
    if code_block.startswith("python"):
        code_block = code_block[6:]

    return code_block

In [43]:
# Module I |||| Practicing Programmatic Prompting for Agents (Solution)
# Me pregunta el nombre que se le quiere poner al archivo
def develop_custom_function():
    print("\nWhat kind of function would you like to create?")
    print("Example: 'A function that calculates the factorial of a number'")
    print("Your description: ", end='')
    function_description = input().strip()

    messages = [
        {"role": "system", "content": "You are a Python expert helping to develop a function."}
    ]

    messages.append({
        "role": "user",
        "content": f"Write a Python function that {function_description}. Output the function in a ```python code block```."
    })
    initial_function = generate_response(messages)
    initial_function = extract_code_block(initial_function)

    print("\n=== Initial Function ===")
    print(initial_function)

    messages.append({"role": "assistant", "content": "```python\n\n"+initial_function+"\n\n```"})

    messages.append({
        "role": "user",
        "content": "Add comprehensive documentation to this function, including description, parameters, "
                   "return value, examples, and edge cases. Output the function in a ```python code block```."
    })
    documented_function = generate_response(messages)
    documented_function = extract_code_block(documented_function)
    print("\n=== Documented Function ===")
    print(documented_function)

    messages.append({"role": "assistant", "content": "```python\n\n"+documented_function+"\n\n```"})

    messages.append({
        "role": "user",
        "content": "Add unittest test cases for this function, including tests for basic functionality, "
                   "edge cases, error cases, and various input scenarios. Output the code in a ```python code block```."
    })
    test_cases = generate_response(messages)
    test_cases = extract_code_block(test_cases)
    print("\n=== Test Cases ===")
    print(test_cases)

    filename = function_description.lower()
    filename = ''.join(c for c in filename if c.isalnum() or c.isspace())
    filename = filename.replace(' ', '_')[:30] + '.py'

    with open(filename, 'w') as f:
        f.write(documented_function + '\n\n' + test_cases)

    return documented_function, test_cases, filename

if __name__ == "__main__":
    function_code, tests, filename = develop_custom_function()
    print(f"\nFinal code has been saved to {filename}")


What kind of function would you like to create?
Example: 'A function that calculates the factorial of a number'
Your description: 
=== Initial Function ===

def funcionModuloI(dividendo, divisor):
    """
    Compute the integer modulo (remainder) of `dividendo` divided by `divisor`.

    Parameters
    ----------
    dividendo : int, float, or any type supporting the % operator
        The number to be divided.
    divisor : int, float, or any type supporting the % operator
        The number by which to divide. Must not be zero.

    Returns
    -------
    result : same type as the operands
        The remainder after division (dividendo % divisor).

    Raises
    ------
    TypeError
        If either argument does not support the modulo operator.
    ZeroDivisionError
        If `divisor` is zero.

    Examples
    --------
    >>> funcionModuloI(10, 3)
    1
    >>> funcionModuloI(10.5, 2)
    0.5
    >>> funcionModuloI(-7, 4)
    1
    """
    # Guard against division by zero


In [ ]:
# Building Your First Agent
# The Agent Loop
while iterations < max_iterations:

    # 1. Construct prompt: Combine agent rules with memory
    prompt = agent_rules + memory

    # 2. Generate response from LLM
    print("Agent thinking...")
    response = generate_response(prompt)
    print(f"Agent response: {response}")

    # 3. Parse response to determine action
    action = parse_action(response)

    result = "Action executed"

    if action["tool_name"] == "list_files":
        result = {"result":list_files()}
    elif action["tool_name"] == "read_file":
        result = {"result":read_file(action["args"]["file_name"])}
    elif action["tool_name"] == "error":
        result = {"error":action["args"]["message"]}
    elif action["tool_name"] == "terminate":
        print(action["args"]["message"])
        break
    else:
        result = {"error":"Unknown action: "+action["tool_name"]}

    print(f"Action result: {result}")

    # 5. Update memory with response and results
    memory.extend([
        {"role": "assistant", "content": response},
        {"role": "user", "content": json.dumps(result)}
    ])

    # 6. Check termination condition
    if action["tool_name"] == "terminate":
        break

    iterations += 1


prompt = agent_rules + memory

# agent_rules = [{
#     "role": "system",
#     "content": """
# You are an AI agent that can perform tasks by using available tools.

# Available tools:
# - list_files() -> List[str]: List all files in the current directory.
# - read_file(file_name: str) -> str: Read the content of a file.
# - terminate(message: str): End the agent loop and print a summary to the user.

# If a user asks about files, list them before reading.

# Every response MUST have an action.
# Respond in this format:

# ```action
# {
#     "tool_name": "insert tool_name",
#     "args": {...fill in any required arguments here...}
# }